# Benchmarking sample representation methods on COVID-19 PBMCs


## The problem

We'll work with the [COMBAT dataset](https://www.kaggle.com/datasets/shitovvladimir/a-blood-atlas-of-covid-19-combat-preprocessed) {cite}`ahern2022blood`: ~140 PBMC samples from COVID-19 patients and healthy donors, processed across multiple hospitals (`Source` / `Institute`) and sequencing pools (`Pool_ID`), with clinical outcomes spanning healthy → mild → severe → critical → death within 28 days.

We want a sample-level representation that lets us answer two questions in one go:

1. **Does it capture the clinical signal?** Patients with worse outcomes should sit closer to each other than to healthy donors.
2. **Is it robust to technical variation?** Whether a sample came from hospital A or hospital B, or pool 1 vs pool 2, should be (almost) invisible — those are technical artefacts, not biology.

These two requirements are in tension. A representation that scores high on (1) by accidentally encoding which institute ran the experiment is no use. In this notebook we'll run several `patpy` methods, evaluate each against both criteria, and end with a single comparison table.


## What we'll do

Starting from the simplest representation (pseudobulk) and working up to deep, batch-aware methods (MrVI, scPoli), we'll:

1. Set up COMBAT and extract sample-level metadata.
2. Run each method through the same `patpy` lifecycle (`prepare_anndata` → `calculate_distance_matrix` → `plot_embedding` → `evaluate_representation`).
3. Score every method against the same set of covariates with kNN classification / regression and with a trajectory-correlation score on `Outcome`.
4. Compare side-by-side in one table, then briefly run the same workflow on a different dataset (HLCA) to show it ports.

### Installation

Most of what we use ships with the base install (`pip install patpy`). The deep methods need extras (and ideally a CUDA GPU):

```bash
pip install patpy[pilot,mrvi,scpoli,diffusionemd]
pip install mofapy2  # MOFA isn't behind an extra yet
```

GloScope's R version has its own setup; see the dedicated tutorial linked at the end. A reproducible conda env is at [`envs/benchmark_tutorial.yaml`](https://github.com/lueckenlab/patpy/blob/main/envs/benchmark_tutorial.yaml).


## Import packages

In [ ]:
import ehrapy as ep
import pandas as pd
import scanpy as sc
import patpy
import seaborn as sns
from plottable import ColumnDefinition, Table
from plottable.cmap import normed_cmap
from plottable.plots import bar
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

In [ ]:
patpy.__version__

## Read the data

We'll start with the [COMBAT dataset](https://www.kaggle.com/datasets/shitovvladimir/a-blood-atlas-of-covid-19-combat-preprocessed). {cite}`ahern2022blood` This dataset contains 783k peripheral blood mononuclear cells from 140 samples of COVID-19 patients and healthy donors. A preprocessed version of COMBAT dataset can be easily loaded with `patpy`:

In [ ]:
adata = patpy.datasets.combat()
adata

Set the `PATPY_NB_TEST` environment variable to a non-empty value before launching the kernel to run a quick smoke-test on a tiny subset (15 donors, 10% of cells per donor with a floor of 10). Otherwise the notebook runs on the full ~783k cells / ~140 donors.


In [ ]:
import os
import numpy as np

if os.environ.get("PATPY_NB_TEST"):
    rng = np.random.default_rng(67)
    sample_ids = adata.obs["scRNASeq_sample_ID"].unique()
    test_donors = rng.choice(sample_ids, size=min(15, len(sample_ids)), replace=False)
    adata = adata[adata.obs["scRNASeq_sample_ID"].isin(test_donors)].copy()
    adata = patpy.pp.subsample(
        adata,
        obs_category_col="scRNASeq_sample_ID",
        min_samples_per_category=10,
        fraction=0.1,
    )
    print(f"PATPY_NB_TEST: subsampled to {adata.n_obs} cells across "
          f"{adata.obs['scRNASeq_sample_ID'].nunique()} donors")


Set columns containing sample IDs, cell types and `.obs` columns containing sample-level metadata:

In [ ]:
sample_key = "scRNASeq_sample_ID"
cell_type_key = "cell_type"
sample_level_columns = ["Source", "Outcome", "Death28", "Institute", "Pool_ID", "Age"]

# Per-method runtimes accumulated as we go; later cells aggregate per dataset.
runtimes = {"combat": {}}
dataset_summaries = {}


Currently, there is no such columns as "cell_type" in the data. But cell types are stored in the `Annotation_major_subset` column. Let's rename it to `cell_type` for better readability.

In [ ]:
adata.obs.rename(columns={"Annotation_major_subset": cell_type_key}, inplace=True)

## Store metadata and calculate QC metrics

We want to evaluate how different sample representation methods preserve the useful information and whether they are affected by batch effects. To do that, we need to extract sample-level metadata and aggregate cell-level QC metrics. All of it can be conveniently done with `patpy` preprocessing module:

In [ ]:
metadata = patpy.pp.extract_metadata(adata, sample_key, sample_level_columns)
metadata

This function will aggregate cell level QC metrics per sample. By default, median aggregation is used. Make sure the columns you want to aggregate are in `adata.obs`! Here, we'll use number of genes per cell, percentage of mitochondrial genes, and doublet score:

In [ ]:
cell_qc_metadata = patpy.pp.calculate_cell_qc_metrics(
    adata, sample_key=sample_key, cell_qc_vars=["QC_ngenes", "QC_pct_mitochondrial", "QC_scrub_doublet_scores"]
)
cell_qc_metadata

In [ ]:
n_genes_metadata = patpy.pp.calculate_n_cells_per_sample(adata, sample_key)
n_genes_metadata

Additionally, let's compute cell type proportions to see if they impact sample representations:

In [ ]:
composition_metadata = patpy.pp.calculate_compositional_metrics(adata, sample_key, [cell_type_key], normalize_to=100)
composition_metadata

Merge metadata tables into one. Always make sure that sample order is the same!

In [ ]:
metadata = pd.concat(
    [
        metadata,
        cell_qc_metadata.loc[metadata.index],
        n_genes_metadata.loc[metadata.index],
        composition_metadata.loc[metadata.index],
    ],
    axis=1,
)

In [ ]:
metadata.shape

## Two questions, one notebook

Before running any methods, let's pin down what "good" looks like. We split the metadata columns into two buckets:

| Type | Covariates | What a good representation does |
| --- | --- | --- |
| Clinical | `Outcome`, `Death28`, `Source` | High kNN score — captures it |
| Technical | `Institute`, `Pool_ID`, `n_cells`, `median_QC_ngenes` | Low kNN score — ignores it |

We'll evaluate every representation against this same set, plus a trajectory-correlation score that asks how well diffusion pseudotime over the representation tracks `Outcome` (rooted at the youngest healthy donor). The final table at the bottom of the notebook scores all methods on all covariates side-by-side.


## Quality control

To reduce noise in the representations, we need to remove samples with too few cells:

In [ ]:
adata.obs[sample_key].value_counts()

In [ ]:
adata = patpy.pp.filter_small_samples(adata, sample_key=sample_key, sample_size_threshold=200)

If necessary, we can also remove cell types with too few cells in at least one sample:

`adata = patpy.pp.filter_small_cell_groups(adata, sample_key=sample_key, cell_group_key=cell_type_key, cluster_size_threshold=10)`

Some methods require this filtering step prior to building representation. In this notebook, we'll focus on simpler methods that can be used with only sample filtering.

## Method 1 — Pseudobulk


### Why this method?

Pseudobulk is the simplest sample representation: average every cell of a sample into a single vector. It's a strong baseline because most signals visible at the tissue level survive averaging. The catch is the choice of cell-level features — raw expression, PCA, or an integrated latent space (scVI, scANVI, scPoli) all give different answers. Below we try three layers and look at what changes.


In [ ]:
# Set up the sample representation method
pseudobulk_gene_expression = patpy.tl.Pseudobulk(
    sample_key=sample_key,
    cell_group_key=cell_type_key,  
    layer="X"  # Use adata.X that currently contains log-normalised expression
)
pseudobulk_gene_expression.prepare_anndata(adata)  # Prepare data
pseudobulk_gene_expression.calculate_distance_matrix(force=True);  # Compute sample representation

But which cell representations should we use? In the literature, there is an inconsistency: some people prefer using gene expression, while other use latent features of the cells produced by dimensionality reduction methods. We can quickly test different cell representations by changing `layer` parameter:

In [ ]:
import time as _time; _t0 = _time.time()
pseudobulk_pca = patpy.tl.Pseudobulk(sample_key=sample_key, cell_group_key=cell_type_key, layer="X_pca")
pseudobulk_pca.prepare_anndata(adata)
pseudobulk_pca.calculate_distance_matrix(force=True);
runtimes["combat"]["pseudobulk"] = _time.time() - _t0


In [ ]:
pseudobulk_scvi = patpy.tl.Pseudobulk(sample_key=sample_key, cell_group_key=cell_type_key, layer="X_scVI_batch")
pseudobulk_scvi.prepare_anndata(adata)
pseudobulk_scvi.calculate_distance_matrix(force=True);

Representations can now easily be visualised:

In [ ]:
pseudobulk_gene_expression.embeddings

In [ ]:
pseudobulk_pca.embeddings

In [ ]:
pseudobulk_gene_expression.plot_embedding("UMAP", sample_level_columns)

In [ ]:
pseudobulk_pca.plot_embedding("UMAP", sample_level_columns)

In [ ]:
pseudobulk_scvi.plot_embedding("UMAP", sample_level_columns)

### Evaluate how well a covariate is represented

Let's try to classify "Outcome" based on the nearest neighbors for each sample in the representation:

In [ ]:
pseudobulk_pca.evaluate_representation(target="Outcome", method="knn", n_neighbors=5, task="classification")

It doesn't work too good. Now we can try to solve the ranking problem for the same covariate. It will train regressor and use a different metric for evaluation:

In [ ]:
pseudobulk_pca.evaluate_representation(target="Outcome", method="knn", n_neighbors=5, task="ranking")

Now let's see how well Pool is represented. This is a technical covariate so we don't want score to be high:

In [ ]:
pseudobulk_pca.evaluate_representation(target="Pool_ID", method="knn", n_neighbors=5, task="classification")

Save the distances to adata to use later

In [ ]:
adata.uns["pseudobulk_distances"] = pseudobulk_pca.calculate_distance_matrix()
adata.uns["pseudobulk_samples"] = pseudobulk_pca.samples
adata.uns["pseudobulk_UMAP"] = pseudobulk_pca.embeddings["UMAP"]

## Method 2 — CellGroupComposition


This representation is based on cell type composition differences. It is calculated as a difference between cell type proportions in each sample.

In [ ]:
import time as _time; _t0 = _time.time()
composition = patpy.tl.CellGroupComposition(
    sample_key=sample_key, cell_group_key=cell_type_key,
)
composition.prepare_anndata(adata)
composition_distances = composition.calculate_distance_matrix(force=True)
runtimes["combat"]["composition"] = _time.time() - _t0


In [ ]:
composition.plot_embedding(method="UMAP", metadata_cols=sample_level_columns)

We can also visualise cell type proportions for each sample:

In [ ]:
composition.sample_representation.plot(kind="bar", stacked=True, figsize=(10, 8), width=0.8)
plt.xticks([])
plt.legend(loc=(1.05, 0), title="Cell type");

We can see that cell type composition reflects patient outcome worse than pseudobulk:

In [ ]:
composition.evaluate_representation(target="Outcome", method="knn", n_neighbors=5, task="ranking")

In [ ]:
adata.uns["composition_distances"] = composition_distances
adata.uns["composition_samples"] = composition.samples

## Method 3 — PILOT


[PILOT](https://pubmed.ncbi.nlm.nih.gov/38177382/) is an Optimal Transport-based tool, which calculates distances between samples based on cell type proportion differences taking into account cell type similarities. Note that to run it, you need to install the dependencies additionally:

`pip install patpy[pilot]`

In [ ]:
import time as _time; _t0 = _time.time()
pilot = patpy.tl.PILOT(
    sample_key=sample_key,
    cell_group_key=cell_type_key,
    layer="X_scVI_batch",
    sample_state_col="Outcome",  # not used for distance calculation
)
runtimes["combat"]["pilot"] = _time.time() - _t0


In [ ]:
pilot.prepare_anndata(adata)

In [ ]:
pilot.calculate_distance_matrix()

In [ ]:
pilot.plot_embedding(method="UMAP", metadata_cols=sample_level_columns);

In [ ]:
adata.uns["pilot_distances"] = pilot.calculate_distance_matrix()
adata.uns["pilot_samples"] = pilot.samples
adata.uns["pilot_UMAP"] = pilot.embeddings["UMAP"]

Let's check the covariates:

In [ ]:
pilot.evaluate_representation(target="Outcome", method="knn", n_neighbors=5, task="ranking")

In [ ]:
pilot.evaluate_representation(target="Pool_ID", method="knn", n_neighbors=5, task="classification")

Outcome is represented a bit worse, but Pool ID almost doesn't affect the representation

## Method 4 — GroupedPseudobulk


### Why this method?

GroupedPseudobulk is one step richer than Pseudobulk: instead of one vector per sample, you get one vector *per cell type per sample*, then concatenate them. If your biological signal lives in *which* cell types respond rather than the global average, this representation surfaces it.

Unlike plain Pseudobulk, this method needs at least a few cells in every (sample, cell_type) pair, so we drop very sparse cell types for this method only (via `patpy.pp.filter_small_cell_groups`). All other methods keep the full set of cell types.


In [ ]:
import time as _time; _t0 = _time.time()
# GroupedPseudobulk iterates per (sample, cell_type) -- drop sparse cell
# types just for this method so the per-cell-type pseudobulks are well
# defined. The full set of cell types stays available for every other method.
adata_for_grouped = patpy.pp.filter_small_cell_groups(
    adata,
    sample_key=sample_key,
    cell_group_key=cell_type_key,
    cluster_size_threshold=5,
)

grouped_pseudobulk = patpy.tl.GroupedPseudobulk(
    sample_key=sample_key,
    cell_group_key=cell_type_key,
    layer="X_scVI_batch",
)
grouped_pseudobulk.prepare_anndata(adata_for_grouped)
grouped_pseudobulk.calculate_distance_matrix();
runtimes["combat"]["grouped_pseudobulk"] = _time.time() - _t0


In [ ]:
grouped_pseudobulk.plot_embedding(method="UMAP", metadata_cols=sample_level_columns);


In [ ]:
adata.uns["grouped_pseudobulk_distances"] = grouped_pseudobulk.calculate_distance_matrix()
adata.uns["grouped_pseudobulk_samples"] = grouped_pseudobulk.samples
adata.uns["grouped_pseudobulk_UMAP"] = grouped_pseudobulk.embeddings["UMAP"]


## Method 5 — RandomVector (sanity baseline)


### Why this method?

RandomVector assigns each sample a vector of Gaussian noise. It exists as a *floor* for the comparison table: any real method should beat random on the clinical covariates and should *not* beat random on the technical ones.


In [ ]:
import time as _time; _t0 = _time.time()
random = patpy.tl.RandomVector(
    sample_key=sample_key,
    cell_group_key=cell_type_key,
)
random.prepare_anndata(adata)
random.calculate_distance_matrix();
runtimes["combat"]["random"] = _time.time() - _t0


In [ ]:
random.plot_embedding(method="UMAP", metadata_cols=sample_level_columns);


In [ ]:
adata.uns["random_distances"] = random.calculate_distance_matrix()
adata.uns["random_samples"] = random.samples
adata.uns["random_UMAP"] = random.embeddings["UMAP"]


## Method 6 — MOFA


### Why this method?

MOFA2 fits a multi-view factor model: each cell type becomes a view, and shared latent factors are inferred across views. The factors capture biological axes of variation that are coherent across cell types — useful when no single cell type tells the whole story.

Requires `pip install mofapy2` (no `[mofa]` extra in patpy yet).


In [ ]:
import time as _time; _t0 = _time.time()
mofa = patpy.tl.MOFA(
    sample_key=sample_key,
    cell_group_key=cell_type_key,
    n_factors=10,
    aggregate_cell_types=True,
)
mofa.prepare_anndata(adata)
mofa.calculate_distance_matrix();
runtimes["combat"]["mofa"] = _time.time() - _t0


In [ ]:
mofa.plot_embedding(method="UMAP", metadata_cols=sample_level_columns);


In [ ]:
adata.uns["mofa_distances"] = mofa.calculate_distance_matrix()
adata.uns["mofa_samples"] = mofa.samples
adata.uns["mofa_UMAP"] = mofa.embeddings["UMAP"]


## Method 7 — GloScope (Python)


### Why this method?

GloScope models each sample as a distribution over the cell-state manifold (in some latent embedding) and computes inter-sample distances as a distribution divergence (default: symmetric KL via k-nearest neighbours). Conceptually it sits between PILOT (which compares cell-type proportions weighted by cell similarity) and DiffusionEMD (which moves mass along the diffusion graph): GloScope ignores cell-type labels entirely and just compares the cells' positions in feature space.

We use the pure-Python implementation (`patpy.tl.GloScope_py`); the canonical R version is benchmarked separately in [`sources_of_variation_with_gloscope.ipynb`](./sources_of_variation_with_gloscope.ipynb). Needs `pip install patpy[gloscope-py-cpu]` (or `patpy[gloscope-py-gpu]` for a CUDA speedup).


In [ ]:
import time as _time; _t0 = _time.time()
gloscope = patpy.tl.GloScope_py(
    sample_key=sample_key,
    cell_group_key=cell_type_key,
    layer="X_scVI_batch",
    k=25,
)
gloscope.prepare_anndata(adata)
gloscope.calculate_distance_matrix();
runtimes["combat"]["gloscope"] = _time.time() - _t0


In [ ]:
gloscope.plot_embedding(method="UMAP", metadata_cols=sample_level_columns);


In [ ]:
adata.uns["gloscope_distances"] = gloscope.calculate_distance_matrix()
adata.uns["gloscope_samples"] = gloscope.samples
adata.uns["gloscope_UMAP"] = gloscope.embeddings["UMAP"]


## Comparing representations

Let's write a small function to put all the sample representations into a single object. It will make sure that the order of samples is identical and cluster the patients

In [ ]:
def align_representations(adata, meta_adata, samples, methods):
    """
    Align representations of different methods to have the same order of samples.

    Additionally runs clustering with Leiden algorithm.
    """
    for method in methods:
        #     samples_to_take = np.isin(adata.uns[f"{method}_samples"], samples)
        representation_samples = list(adata.uns[f"{method}_samples"])
        samples_order = [representation_samples.index(sample) for sample in samples if sample in representation_samples]

        assert (np.array(adata.uns[f"{method}_samples"])[samples_order] == samples).all(), "Order of samples is not correct"

        # meta_adata.obsm["umap"] = meta_adata.obsm[f"{method}_UMAP"]
        meta_adata.obsm[f"{method}_distances"] = adata.uns[f"{method}_distances"][samples_order][:, samples_order]

        ep.pp.neighbors(
            meta_adata, use_rep=f"{method}_distances", key_added=f"{method}_neighbors", metric="precomputed"
        )
        ep.tl.leiden(meta_adata, key_added=f"{method}_leiden", neighbors_key=f"{method}_neighbors")

    return meta_adata

In [ ]:
combat_methods = [
    "pseudobulk",
    "composition",
    "pilot",
    "grouped_pseudobulk",
    "random",
    "mofa",
    "gloscope",
]


In [ ]:
combat_samples = list(set(adata.uns[f"{method}_samples"]) for method in combat_methods)
combat_samples = list(set.intersection(*combat_samples))
len(combat_samples)

In [ ]:
metadata = metadata.loc[combat_samples]

In [ ]:
combat_meta_adata = ep.io.df_to_anndata(metadata)
combat_meta_adata = ep.pp.encode(combat_meta_adata, autodetect=True)
combat_meta_adata

In [ ]:
combat_meta_adata = align_representations(
    adata=adata,
    meta_adata=combat_meta_adata,
    samples=combat_samples,
    methods=combat_methods,
)

## Evaluation 1 — kNN scores per covariate

For each (representation, covariate) pair, we run a k-nearest-neighbours classification or regression directly on the precomputed sample distances. Scores are normalised to `[0, 1]` and we *invert* the score for technical covariates so that 1 = good (representation ignores the technical artefact) in every column.


In [ ]:
benchmark_schema = {
    "technical": ["Institute", "Pool_ID", "n_cells", "median_QC_ngenes"],
    "clinical": ["Death28", "Outcome", "Source"],
}

cols_with_tasks = {
    "Institute": "classification",
    "Pool_ID": "classification",
    "n_cells": "regression",
    "median_QC_ngenes": "regression",
    "Death28": "classification",
    "Outcome": "ranking",
    "Source": "classification",
}

# One colour per method, used in every plot below so the same method has the
# same colour wherever it appears.
method_colors = {
    "pseudobulk": "#1f77b4",
    "composition": "#ff7f0e",
    "pilot": "#2ca02c",
    "grouped_pseudobulk": "#d62728",
    "random": "#7f7f7f",
    "mofa": "#9467bd",
    "gloscope": "#bcbd22",
}


In [ ]:
results = []

for method in combat_methods:
    for covariate_type in benchmark_schema:
        for col in benchmark_schema[covariate_type]:
            task = cols_with_tasks[col]
            try:
                result = patpy.tl.evaluate_representation(
                    distances=combat_meta_adata.obsm[f"{method}_distances"],
                    target=metadata[col],
                    method="knn",
                    task=task,
                    #             n_neighbors=5
                )
            except Exception as e:
                print("Method:", method)
                print("Col:", col)
                print("Task:", task)
                print(e)
                print()
                continue
            #             raise(e)
            result["representation"] = method
            result["covariate"] = col
            result["covariate_type"] = covariate_type

            if result["metric"] == "spearman_r":
                result["score"] = abs(result["score"])

            if covariate_type == "technical":
                result["score"] = 1 - result["score"]

            results.append(result)

Scores are ranged from 0 to 1

- For covariates that we defined as technical, 0 means that covariate strongly affects the representation, and 1 means that this covariate is randomly distributed across representation
- For biological and clinical covariates, 0 means that a covariate is not represented well (it is randomly distributed), while 1 means that similar patients have similar values of covariate 

In [ ]:
knn_results = pd.DataFrame(results)
knn_results.sort_values("score", ascending=False)

In [ ]:
# plt.figure(figsize=(10, 20))
sns.barplot(data=knn_results, y="covariate", x="score", orient="h", hue="representation", palette=method_colors)
plt.xlim(0, 1.05)
plt.title("KNN-score", fontsize=24)

## Evaluation 2 — trajectory preservation

kNN scores are local — they ask whether each sample's neighbours share the same covariate value. They don't tell us whether the *order* of severity is preserved: do healthy donors sit on one side, deceased patients on the other, and milder cases in between?

We answer that with diffusion pseudotime, rooted at a healthy donor: a good representation should produce a pseudotime that correlates with `Outcome` (higher = worse outcome). The Spearman correlation per representation is the score we add to the comparison table.


In [ ]:
# Trajectory root: the donor closest to the healthy end of the COMBAT severity scale.
# In COMBAT, Outcome is coded so 6 = healthy / mild HCW and 1 = critical, so we sort
# by Outcome descending. Among the healthiest donors, we break ties on Age (youngest).
# This is robust to test-mode subsampling that may drop all Source=='HV' samples.
candidates = metadata.dropna(subset=["Outcome", "Age"])
root_sample = candidates.sort_values(["Outcome", "Age"], ascending=[False, True]).index[0]
print(
    f"Trajectory root: {root_sample} "
    f"(Outcome={metadata.loc[root_sample, 'Outcome']:.0f}, "
    f"Age={metadata.loc[root_sample, 'Age']:.0f}, "
    f"Source={metadata.loc[root_sample, 'Source']})"
)


In [ ]:
trajectory_results = patpy.tl.trajectory_correlation(
    meta_adata=combat_meta_adata,
    root_sample=root_sample,
    trajectory_variable="Outcome",
    representations=combat_methods,
    inverse_trajectory=True,  # COMBAT codes 6 = healthy, 1 = critical
)
trajectory_results


In [ ]:
knn_results_wide = knn_results.pivot(index="representation", columns="covariate", values="score")

# Add trajectory preservation score early so it can feed into col_defs below
knn_results_wide["Trajectory"] = (
    trajectory_results.loc[knn_results_wide.index, "correlation"].abs()
)

# Group covariates by the two scoring buckets, with new display names
display_buckets = {
    "Information retention": benchmark_schema["clinical"],
    "Batch mixing": benchmark_schema["technical"],
}

# Per-bucket means -> bucket columns
for bucket, cols in display_buckets.items():
    knn_results_wide[bucket] = knn_results_wide[cols].mean(axis=1)

# New total: (2 * info + 2 * trajectory + batch_mixing) / 5
knn_results_wide["Total"] = (
    2 * knn_results_wide["Information retention"]
    + 2 * knn_results_wide["Trajectory"]
    + knn_results_wide["Batch mixing"]
) / 5

cols_order = ["Total", "Information retention", "Trajectory", "Batch mixing"]
cols_order += display_buckets["Information retention"]
cols_order += display_buckets["Batch mixing"]

cmap_bar = LinearSegmentedColormap.from_list(
    name="bugw", colors=["#FF9693", "#f2fbd2", "#c9ecb4", "#93d3ab", "#35b0ab"], N=256,
)

bar_kw = {
    "cmap": cmap_bar, "plot_bg_bar": True, "annotate": True,
    "height": 0.5, "lw": 0.5, "formatter": lambda x: round(x, 2),
}

col_defs = [
    ColumnDefinition("Total", width=0.7, plot_fn=bar, plot_kw=bar_kw),
    ColumnDefinition(
        "Information retention", width=0.8, plot_fn=bar, plot_kw=bar_kw,
        group="aggregate",
    ),
    ColumnDefinition(
        "Trajectory", width=0.7, plot_fn=bar, plot_kw=bar_kw,
        group="aggregate",
    ),
    ColumnDefinition(
        "Batch mixing", width=0.7, plot_fn=bar, plot_kw=bar_kw,
        group="aggregate",
    ),
]
for col in display_buckets["Information retention"]:
    col_defs.append(ColumnDefinition(
        name=col, width=0.55, formatter=lambda x: round(x, 2),
        textprops={"ha": "center", "bbox": {"boxstyle": "circle", "pad": 0.35}},
        cmap=normed_cmap(knn_results["score"], cmap=matplotlib.cm.PiYG, num_stds=2.5),
        group="Information retention covariates",
    ))
for col in display_buckets["Batch mixing"]:
    col_defs.append(ColumnDefinition(
        name=col, width=0.55, formatter=lambda x: round(x, 2),
        textprops={"ha": "center", "bbox": {"boxstyle": "circle", "pad": 0.35}},
        cmap=normed_cmap(knn_results["score"], cmap=matplotlib.cm.PiYG, num_stds=2.5),
        group="Batch mixing covariates",
    ))


In [ ]:
fig, ax = plt.subplots(figsize=(20, 6))
Table(knn_results_wide[cols_order].sort_values("Total", ascending=False), column_definitions=tuple(col_defs), ax=ax)


### How to read this table

Every cell is a score in `[0, 1]` where **higher = better**:

- **Clinical** columns (`Outcome`, `Death28`, `Source`): high means a kNN classifier trained on the sample distances recovers the clinical label.
- **Technical** columns (`Institute`, `Pool_ID`, `n_cells`, `median_QC_ngenes`): high means the *opposite* — the score has been inverted so that high indicates the representation does **not** encode the technical artefact.
- **`trajectory`** is the absolute Spearman correlation between diffusion pseudotime (rooted at the healthiest donor) and `Outcome`. High means the order of severity is preserved along the representation's diffusion graph.
- **`total`** is a weighted average that gives more weight to clinical signal than to batch invariance, so the top of the sorted table is the method most useful for downstream clinical analysis on this dataset.

The `random` baseline is the floor: any real method should beat it on clinical and trajectory columns. If a method *doesn't* beat random there, the method or its hyperparameters need work for this dataset.


## A reusable benchmark helper

To compare against other datasets we need to repeat the same workflow: run every method, score each one against a "relevant" (clinical) covariate set and a "technical" covariate set, and compute a trajectory correlation. Below is a single helper that does all of that, returning per-method distances, kNN scores, and a summary table. We'll call it three times: once each for COMBAT (already done above), HLCA, and Stephenson.


In [ ]:
import time
import gc

def run_all_methods_on_dataset(
    adata,
    *,
    sample_key,
    cell_type_key,
    layer="X_scVI_batch",
    cluster_size_threshold=5,
):
    """Run all seven representation methods on `adata`.

    Returns
    -------
    dict with keys 'distances', 'samples', 'runtimes' (each keyed by method name).
    """
    out = {"distances": {}, "samples": {}, "runtimes": {}}

    method_specs = [
        ("pseudobulk", lambda: patpy.tl.Pseudobulk(
            sample_key=sample_key, cell_group_key=cell_type_key, layer=layer)),
        ("composition", lambda: patpy.tl.CellGroupComposition(
            sample_key=sample_key, cell_group_key=cell_type_key)),
        ("pilot", lambda: patpy.tl.PILOT(
            sample_key=sample_key, cell_group_key=cell_type_key, layer=layer)),
        ("random", lambda: patpy.tl.RandomVector(
            sample_key=sample_key, cell_group_key=cell_type_key)),
        ("mofa", lambda: patpy.tl.MOFA(
            sample_key=sample_key, cell_group_key=cell_type_key,
            n_factors=10, aggregate_cell_types=True)),
        ("gloscope", lambda: patpy.tl.GloScope_py(
            sample_key=sample_key, cell_group_key=cell_type_key, layer=layer, k=25)),
    ]
    for name, factory in method_specs:
        try:
            t0 = time.time()
            m = factory()
            m.prepare_anndata(adata)
            out["distances"][name] = m.calculate_distance_matrix(force=True)
            out["samples"][name] = list(m.samples)
            out["runtimes"][name] = time.time() - t0
            print(f"  {name}: {out['runtimes'][name]:.1f}s")
        except Exception as e:
            print(f"  {name} FAILED: {type(e).__name__}: {e}")

    # GroupedPseudobulk needs a denser per-(sample, cell_type) matrix --
    # filter sparse cell types just for this method.
    try:
        adata_filtered = patpy.pp.filter_small_cell_groups(
            adata, sample_key=sample_key, cell_group_key=cell_type_key,
            cluster_size_threshold=cluster_size_threshold,
        )
        t0 = time.time()
        m = patpy.tl.GroupedPseudobulk(
            sample_key=sample_key, cell_group_key=cell_type_key, layer=layer)
        m.prepare_anndata(adata_filtered)
        out["distances"]["grouped_pseudobulk"] = m.calculate_distance_matrix()
        out["samples"]["grouped_pseudobulk"] = list(m.samples)
        out["runtimes"]["grouped_pseudobulk"] = time.time() - t0
        print(f"  grouped_pseudobulk: {out['runtimes']['grouped_pseudobulk']:.1f}s")
        del adata_filtered
        gc.collect()
    except Exception as e:
        print(f"  grouped_pseudobulk FAILED: {type(e).__name__}: {e}")

    return out


def score_methods_on_dataset(
    bench_result,
    *,
    metadata,
    sample_key,
    schema,
    root_sample,
    trajectory_variable,
    inverse_trajectory=False,
    n_neighbors=5,
):
    """Score the methods produced by `run_all_methods_on_dataset` against
    a SPARE-style `schema` (dict with at least 'relevant' / 'technical' keys
    mapping to {covariate: task}). Computes a trajectory_correlation if the
    `trajectory_variable` and `root_sample` are present in `metadata`.

    Returns
    -------
    dict with keys 'long' (long-format scores), 'summary' (per-method
    Information retention / Batch mixing / Trajectory / Total / Runtime),
    and 'meta_adata' (the encoded ehrapy AnnData used for diffmap).
    """
    methods = list(bench_result["distances"].keys())
    # Common samples across all methods
    method_sample_sets = {m: set(bench_result["samples"][m]) for m in methods}
    common = set.intersection(*method_sample_sets.values())
    common = [s for s in metadata.index if s in common]
    print(f"  scoring on {len(common)} samples (methods: {[(m, len(s)) for m, s in method_sample_sets.items()]})")
    if len(common) < 4:
        print("  too few common samples for kNN scoring; returning empty summary")
        empty = pd.Series(np.nan, index=methods)
        summary = pd.DataFrame({
            "Information retention": empty,
            "Batch mixing": empty,
            "Trajectory": empty,
            "Runtime (s)": pd.Series(bench_result["runtimes"]).reindex(methods),
            "Total": empty,
        })
        return {"long": pd.DataFrame(), "summary": summary, "meta_adata": None}
    meta = metadata.loc[common].copy()
    # Drop all-NaN columns -- ehrapy.encode chokes on them
    meta = meta.dropna(axis=1, how="all")
    meta_adata = ep.io.df_to_anndata(meta)
    meta_adata = ep.pp.encode(meta_adata, autodetect=True)

    for m in methods:
        order = [bench_result["samples"][m].index(s) for s in common]
        d = bench_result["distances"][m]
        meta_adata.obsm[f"{m}_distances"] = d[np.ix_(order, order)]
        _n_nbrs = min(15, max(2, len(common) - 1))
        ep.pp.neighbors(
            meta_adata, use_rep=f"{m}_distances",
            key_added=f"{m}_neighbors", metric="precomputed",
            n_neighbors=_n_nbrs,
        )

    # kNN scoring
    long_rows = []
    for group, cov_tasks in schema.items():
        for col, task in cov_tasks.items():
            if col not in meta.columns:
                continue
            for m in methods:
                try:
                    r = patpy.tl.evaluate_representation(
                        distances=meta_adata.obsm[f"{m}_distances"],
                        target=meta[col],
                        method="knn", task=task, n_neighbors=n_neighbors,
                    )
                except Exception as e:
                    print(f"  skipped {m}/{col}: {e}")
                    continue
                score = r["score"]
                if r.get("metric") == "spearman_r":
                    score = abs(score)
                if group == "technical":
                    score = 1 - score
                long_rows.append({
                    "representation": m, "covariate": col,
                    "covariate_type": group, "score": score,
                })
    long_df = pd.DataFrame(long_rows)

    # Trajectory correlation
    traj_series = pd.Series(np.nan, index=methods)
    if (
        trajectory_variable in meta.columns
        and root_sample in meta.index
    ):
        # If trajectory_variable is categorical text, encode to ordered numeric
        traj_target = meta[trajectory_variable]
        if traj_target.dtype.name == "category" or traj_target.dtype == object:
            traj_target_num = traj_target.astype("category").cat.codes.astype(float)
            traj_target_num[traj_target.isna()] = np.nan
            meta_adata.obs[trajectory_variable] = traj_target_num.values
        try:
            traj_df = patpy.tl.trajectory_correlation(
                meta_adata=meta_adata, root_sample=root_sample,
                trajectory_variable=trajectory_variable, representations=methods,
                inverse_trajectory=inverse_trajectory,
            )
            traj_series = traj_df["correlation"].abs()
        except Exception as e:
            print(f"  trajectory_correlation failed: {e}")
    else:
        print(
            f"  trajectory skipped: variable={trajectory_variable!r} in metadata? "
            f"{trajectory_variable in meta.columns}; root={root_sample!r} in samples? "
            f"{root_sample in meta.index}"
        )

    # Per-method summary
    methods_idx = pd.Index(methods, name="representation")
    info = pd.Series({
        m: long_df[
            (long_df["representation"] == m)
            & (long_df["covariate_type"].isin(["relevant", "clinical"]))
        ]["score"].mean()
        for m in methods
    })
    batch = pd.Series({
        m: long_df[
            (long_df["representation"] == m)
            & (long_df["covariate_type"] == "technical")
        ]["score"].mean()
        for m in methods
    })
    traj = traj_series.reindex(methods_idx)
    runtimes_s = pd.Series(bench_result["runtimes"]).reindex(methods_idx)

    summary = pd.DataFrame({
        "Information retention": info,
        "Batch mixing": batch,
        "Trajectory": traj,
        "Runtime (s)": runtimes_s,
    })
    # New total: (2*info + 2*traj + batch) / 5; NaN trajectory falls back to 0 weight
    traj_filled = summary["Trajectory"].fillna(0)
    summary["Total"] = (
        2 * summary["Information retention"].fillna(0)
        + 2 * traj_filled
        + summary["Batch mixing"].fillna(0)
    ) / 5
    summary = summary.sort_values("Total", ascending=False)

    return {"long": long_df, "summary": summary, "meta_adata": meta_adata}


### COMBAT summary for the cross-dataset view


In [ ]:
# Pre-compute a COMBAT summary in the same shape we'll build for HLCA + Stephenson
combat_summary = pd.DataFrame({
    "Information retention": knn_results_wide["Information retention"],
    "Batch mixing": knn_results_wide["Batch mixing"],
    "Trajectory": knn_results_wide["Trajectory"],
    "Runtime (s)": pd.Series(runtimes["combat"]).reindex(knn_results_wide.index),
})
combat_summary["Total"] = knn_results_wide["Total"]
combat_summary = combat_summary.sort_values("Total", ascending=False)
dataset_summaries["combat"] = combat_summary
combat_summary.round(2)


In [ ]:
# COMBAT analysis done; release the AnnData (and any heavy intermediates) so
# HLCA + Stephenson have memory headroom.
del adata, combat_meta_adata, knn_results, knn_results_wide
gc.collect()


## Benchmark on HLCA


On to the Human Lung Cell Atlas (HLCA) {cite}`sikkema2023integrated`: ~340 donors across health and lung disease, multiple sequencing platforms. We subsample to 50 donors x 20% cells so each method finishes in a few minutes, then run the same seven methods with the SPARE benchmark schema (clinical / technical / trajectory). The summary table at the end is structurally identical to COMBAT's, just on a different cohort and different question.


In [ ]:
hlca, hlca_info = patpy.datasets.hlca(return_dataset_info=True)
print(f"HLCA loaded: {hlca.n_obs} cells, {hlca.obs[hlca_info.sample_key].nunique()} donors")


In [ ]:
# In test mode keep this small to iterate fast. Otherwise use the full HLCA --
# all 339 donors, 1.7M cells (no subsampling).
if os.environ.get('PATPY_NB_TEST'):
    rng = np.random.default_rng(67)
    _donors = rng.choice(
        hlca.obs[hlca_info.sample_key].unique(),
        size=min(20, hlca.obs[hlca_info.sample_key].nunique()),
        replace=False,
    )
    hlca = hlca[hlca.obs[hlca_info.sample_key].isin(_donors)].copy()
    hlca = patpy.pp.subsample(
        hlca,
        obs_category_col=hlca_info.sample_key,
        min_samples_per_category=200,
        fraction=0.1,
    )
hlca = patpy.pp.filter_small_samples(hlca, sample_key=hlca_info.sample_key, sample_size_threshold=200)
print(f'HLCA after QC: {hlca.n_obs} cells, {hlca.obs[hlca_info.sample_key].nunique()} donors')


In [ ]:
# Extract sample-level metadata for the SPARE-style scoring (clinical = "relevant",
# technical = batch / acquisition artefacts). Skip "contextual" columns -- we don't
# score them, they're neither clinical signal nor technical noise.
hlca_sample_cols = [
    "tissue", "anatomical_region_ccf_score", "lung_condition", "disease",
    "smoking_status", "suspension_type", "core_or_extension", "fresh_or_frozen",
    "sequencing_platform", "subject_type", "assay", "development_stage",
    "BMI", "age_or_mean_of_age_range", "sex",
]
# Only ask for columns that exist
hlca_sample_cols = [c for c in hlca_sample_cols if c in hlca.obs.columns]
hlca_meta = patpy.pp.extract_metadata(hlca, hlca_info.sample_key, hlca_sample_cols)
hlca_qc = patpy.pp.calculate_cell_qc_metrics(
    hlca, sample_key=hlca_info.sample_key,
    cell_qc_vars=[c for c in ["n_genes_by_counts", "total_counts", "pct_counts_mt"] if c in hlca.obs.columns],
)
hlca_ncells = patpy.pp.calculate_n_cells_per_sample(hlca, hlca_info.sample_key)
hlca_meta = pd.concat([hlca_meta, hlca_qc.loc[hlca_meta.index], hlca_ncells.loc[hlca_meta.index]], axis=1)
hlca_meta.head()


In [ ]:
print("Running 7 methods on HLCA:")
hlca_bench = run_all_methods_on_dataset(
    hlca,
    sample_key=hlca_info.sample_key,
    cell_type_key=hlca_info.cell_type_key,
    layer="X_scVI_batch",
)
runtimes["hlca"] = hlca_bench["runtimes"]


In [ ]:
# SPARE schema for HLCA (https://github.com/lueckenlab/SPARE/blob/main/data/hlca/benchmark_schema.json)
hlca_schema = {
    "relevant": {
        "tissue": "classification",
        "anatomical_region_ccf_score": "regression",
        "lung_condition": "classification",
        "disease": "classification",
        "smoking_status": "classification",
    },
    "technical": {
        "suspension_type": "classification",
        "core_or_extension": "classification",
        "fresh_or_frozen": "classification",
        "sequencing_platform": "classification",
        "subject_type": "classification",
        "assay": "classification",
        "median_n_genes_by_counts": "regression",
        "median_total_counts": "regression",
        "median_pct_counts_mt": "regression",
        "n_cells": "regression",
        "development_stage": "classification",
    },
}

hlca_scored = score_methods_on_dataset(
    hlca_bench,
    metadata=hlca_meta,
    sample_key=hlca_info.sample_key,
    schema=hlca_schema,
    root_sample="homosapiens_None_2023_None_sikkemalisa_002_d10_1101_2022_03_10_483747D322",
    trajectory_variable="anatomical_region_ccf_score",
    inverse_trajectory=False,
)
dataset_summaries["hlca"] = hlca_scored["summary"]
hlca_scored["summary"].round(2)


In [ ]:
# Free HLCA to keep peak memory low before Stephenson loads
del hlca, hlca_bench, hlca_scored
gc.collect()


## Benchmark on Stephenson


A third cohort to show the workflow generalises beyond two datasets. Stephenson et al. {cite}`stephenson2021single` is another COVID-19 PBMC study, with 131 donors collected across three UK sites -- so a different clinical schema (`Worst_Clinical_Status`, `Status_on_day_collection_summary`) and a different technical artefact (`Site`).


In [ ]:
stephenson, stephenson_info = patpy.datasets.stephenson(return_dataset_info=True)
if os.environ.get('PATPY_NB_TEST'):
    _rng = np.random.default_rng(67)
    _donors = _rng.choice(
        stephenson.obs[stephenson_info.sample_key].unique(),
        size=min(20, stephenson.obs[stephenson_info.sample_key].nunique()),
        replace=False,
    )
    stephenson = stephenson[stephenson.obs[stephenson_info.sample_key].isin(_donors)].copy()
    stephenson = patpy.pp.subsample(
        stephenson,
        obs_category_col=stephenson_info.sample_key,
        min_samples_per_category=200,
        fraction=0.1,
    )
print(f'Stephenson: {stephenson.n_obs} cells, {stephenson.obs[stephenson_info.sample_key].nunique()} donors')


In [ ]:
stephenson_sample_cols = [
    "Status", "Status_on_day_collection_summary", "Days_from_onset",
    "Worst_Clinical_Status", "Outcome", "disease", "Site",
    "Swab_result", "Smoker", "sex", "development_stage",
]
stephenson_sample_cols = [c for c in stephenson_sample_cols if c in stephenson.obs.columns]
stephenson = patpy.pp.filter_small_samples(stephenson, sample_key=stephenson_info.sample_key, sample_size_threshold=200)
stephenson_meta = patpy.pp.extract_metadata(stephenson, stephenson_info.sample_key, stephenson_sample_cols)
stephenson_qc = patpy.pp.calculate_cell_qc_metrics(
    stephenson, sample_key=stephenson_info.sample_key,
    cell_qc_vars=[c for c in ["n_genes_by_counts", "total_counts", "pct_counts_mt"] if c in stephenson.obs.columns],
)
stephenson_ncells = patpy.pp.calculate_n_cells_per_sample(stephenson, stephenson_info.sample_key)
stephenson_meta = pd.concat(
    [stephenson_meta, stephenson_qc.loc[stephenson_meta.index], stephenson_ncells.loc[stephenson_meta.index]],
    axis=1,
)
print(f"Stephenson metadata shape: {stephenson_meta.shape}")


In [ ]:
print("Running 7 methods on Stephenson:")
stephenson_bench = run_all_methods_on_dataset(
    stephenson,
    sample_key=stephenson_info.sample_key,
    cell_type_key=stephenson_info.cell_type_key,
    layer="X_scVI_batch",
)
runtimes["stephenson"] = stephenson_bench["runtimes"]


In [ ]:
# SPARE schema for Stephenson (https://github.com/lueckenlab/SPARE/blob/main/data/stephenson/benchmark_schema.json)
stephenson_schema = {
    "relevant": {
        "Status": "classification",
        "Status_on_day_collection_summary": "classification",
        "Days_from_onset": "regression",
        "Worst_Clinical_Status": "classification",
        "Outcome": "classification",
        "disease": "classification",
    },
    "technical": {
        "Site": "classification",
        "median_n_genes_by_counts": "regression",
        "median_total_counts": "regression",
        "median_pct_counts_mt": "regression",
        "n_cells": "regression",
    },
}

stephenson_scored = score_methods_on_dataset(
    stephenson_bench,
    metadata=stephenson_meta,
    sample_key=stephenson_info.sample_key,
    schema=stephenson_schema,
    root_sample="MH8919226",
    trajectory_variable="Status_on_day_collection_summary",
    inverse_trajectory=False,
)
dataset_summaries["stephenson"] = stephenson_scored["summary"]
stephenson_scored["summary"].round(2)


In [ ]:
del stephenson, stephenson_bench, stephenson_scored
gc.collect()


## Cross-dataset summary


Same workflow, three datasets, three sets of scores. Below we average each method's score across the three datasets to get a single number per method per metric, then plot the trade-off: information retention vs batch mixing, with trajectory preservation encoded as marker size and colour. The closer a method is to the top-right corner of the scatter plot, the better it scores on both criteria.


In [ ]:
def _stack_summaries(summaries):
    """Concat per-dataset summaries into a long-format DataFrame."""
    frames = []
    for name, df in summaries.items():
        f = df.copy()
        f["dataset"] = name
        f.index.name = "representation"
        frames.append(f.reset_index())
    return pd.concat(frames, ignore_index=True)


def _mean_summary(summaries):
    """Average each metric across datasets for each method."""
    long = _stack_summaries(summaries)
    agg = long.groupby("representation")[
        ["Information retention", "Batch mixing", "Trajectory", "Total", "Runtime (s)"]
    ].mean()
    return agg.sort_values("Total", ascending=False)


cross_summary = _mean_summary(dataset_summaries)
cross_summary.round(2)


In [ ]:
from matplotlib.lines import Line2D

# Best-effort label placement that avoids overlap: try adjustText if available,
# otherwise fall back to simple per-point offsets that we tune manually.
try:
    from adjustText import adjust_text  # noqa: F401
    HAS_ADJUSTTEXT = True
except ImportError:
    HAS_ADJUSTTEXT = False

fig, ax = plt.subplots(figsize=(8, 6))

# Marker size scaled by trajectory; colour mirrors size for redundancy
traj = cross_summary["Trajectory"].fillna(0)
sizes = 80 + 600 * (traj - traj.min()) / max(traj.max() - traj.min(), 1e-6)
colors = traj.values

sc = ax.scatter(
    cross_summary["Information retention"],
    cross_summary["Batch mixing"],
    s=sizes,
    c=colors,
    cmap="viridis",
    alpha=0.85,
    edgecolor="black",
    linewidth=0.5,
    zorder=3,
)

texts = []
for method, row in cross_summary.iterrows():
    texts.append(ax.text(
        row["Information retention"], row["Batch mixing"], method,
        fontsize=10, ha="left", va="bottom", zorder=4,
    ))

if HAS_ADJUSTTEXT:
    adjust_text(texts, ax=ax, only_move={"text": "xy"},
                arrowprops=dict(arrowstyle="-", color="gray", lw=0.5))

cbar = plt.colorbar(sc, ax=ax, shrink=0.85, pad=0.02)
cbar.set_label("Trajectory preservation\n(|Spearman| pseudotime vs trajectory)", fontsize=9)

ax.set_xlabel("Information retention (clinical kNN, mean over datasets)", fontsize=11)
ax.set_ylabel("Batch mixing (1 - technical kNN, mean over datasets)", fontsize=11)
ax.set_title("Method trade-off averaged across COMBAT, HLCA, and Stephenson", fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()


### Runtime per method per dataset (seconds)


In [ ]:
runtime_table = pd.DataFrame(runtimes).round(1)
runtime_table["mean"] = runtime_table.mean(axis=1).round(1)
runtime_table.sort_values("mean")


Reading the scatter: top-right = best on both criteria. Marker size + colour encode the trajectory score, so a large bright marker in the top-right is a method that captures clinical signal, ignores technical artefacts, *and* preserves the disease trajectory. The runtime table above shows what that costs you in compute.


## Other methods and tutorials

We covered Pseudobulk, GroupedPseudobulk, CellGroupComposition, PILOT, MOFA, GloScope (Python), and a RandomVector baseline. `patpy` also ships:

- **`patpy.tl.MrVI`** — deep generative model conditioned on `sample_key` (and optionally `batch_key`). Best on a CUDA GPU; the torch wheels need to match the system NVIDIA driver version, otherwise it crashes at training time.
- **`patpy.tl.SCPoli`** — conditional VAE that learns a per-sample prototype, batch-aware by construction. Needs `pip install patpy[scpoli]` and (currently) an `anndata < 0.12` env because of how `scarches` imports.
- **`patpy.tl.DiffusionEarthMoverDistance`** — diffusion EMD over composition. Needs `pip install patpy[diffusionemd]` and `scikit-learn < 1.5` for the underlying `DiffusionEMD` package.
- **`patpy.tl.PhEMD`** — pure Python EMD with phenotype trees.
- **`patpy.tl.WassersteinTSNE`** — Wasserstein distances + t-SNE on top.
- **`patpy.tl.GloScope`** — R-based canonical implementation; deeper coverage in [`sources_of_variation_with_gloscope.ipynb`](./sources_of_variation_with_gloscope.ipynb).

Related tutorials:

- [`representation_methods_example.ipynb`](./representation_methods_example.ipynb) — another walkthrough of representation methods on COMBAT.
- [`sources_of_variation_with_gloscope.ipynb`](./sources_of_variation_with_gloscope.ipynb) — deep-dive on GloScope, including the R setup.
- [`supervised_methods_example.ipynb`](./supervised_methods_example.ipynb) — supervised counterparts (MixMIL, PULSAR, PaSCient).
- [`differential_analysis.ipynb`](./differential_analysis.ipynb) — once you have a representation, where to go next.

In the [SPARE benchmark](https://github.com/lueckenlab/SPARE/tree/feature/viash_nextflow_pipeline), GloScope was the best performer overall. The conda env file at [`envs/gloscope.yaml`](https://github.com/lueckenlab/patpy/blob/main/envs/gloscope.yaml) is the simplest way to set up its R dependencies.
